# 🔥 OTAKU UPSCALER V1 — TOPAZ INSPIRED

AI image upscaler for anime and movies.

**Workflow:** Upload → Choose 2×/4× → AI Upscale → Preview → Download

Powered by Real-ESRGAN with a Topaz-inspired enhancement roadmap.

In [ ]:
# ============================================
# 🔥 OTAKU UPSCALER V1 — GPU CHECK
# ============================================

!nvidia-smi

print('🔥 OTAKU UPSCALER V1')
print('====================')
print('GPU environment detected.')

In [ ]:
# ============================================
# 🧠 OTAKU AI ENVIRONMENT
# ============================================

!apt-get -qq update
!apt-get -qq install python3.11 python3.11-venv
!python3.11 -m venv /content/venv311

!/content/venv311/bin/python -m ensurepip --upgrade
!/content/venv311/bin/python -m pip install --upgrade pip
!/content/venv311/bin/pip install -q basicsr facexlib gfpgan

%cd /content
!rm -rf /content/realesrgan
!git clone -q https://github.com/xinntao/Real-ESRGAN.git realesrgan

# BasicSR compatibility patch
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /content/venv311/lib/python3.11/site-packages/basicsr/data/degradations.py

print('✅ OTAKU AI environment ready')

In [ ]:
%%writefile /content/realesrgan/realesrgan/version.py

__version__ = "0.3.0"

In [ ]:
# ============================================
# 📦 REAL-ESRGAN MODEL
# ============================================

!mkdir -p /content/realesrgan/weights
!rm -f /content/realesrgan/weights/RealESRGAN_x4plus.pth

!wget -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O /content/realesrgan/weights/RealESRGAN_x4plus.pth

!ls -lh /content/realesrgan/weights/RealESRGAN_x4plus.pth

print('🔥 Real-ESRGAN model ready')

In [ ]:
# ============================================
# 📁 OTAKU FILE UPLOAD
# ============================================

from google.colab import files
from pathlib import Path

input_folder = Path('/content/input')
output_folder = Path('/content/output')

input_folder.mkdir(exist_ok=True)
output_folder.mkdir(exist_ok=True)

uploaded = files.upload()

for filename, data in uploaded.items():
    (input_folder / filename).write_bytes(data)

print('\n🔥 Uploaded files:')

for filename in uploaded:
    print('✓', filename)

In [ ]:
# ============================================
# ⚡ OTAKU UPSCALE ENGINE
# ============================================

import subprocess
        from pathlib import Path
from PIL import Image
from IPython.display import display

image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

input_files = [
    p for p in input_folder.iterdir()
    if p.suffix.lower() in image_extensions
]

if not input_files:
    raise Exception('❌ No supported image found. Run the upload cell first.')

print('🔥 OTAKU UPSCALER V1')
print('======================')
print('1 → 2× Upscale')
print('2 → 4× Upscale')

choice = input('\nChoose upscale level (1 or 2): ').strip()

if choice == '1':
    scale = 2
elif choice == '2':
    scale = 4
else:
    raise ValueError('❌ Invalid choice. Enter 1 or 2.')

for p in output_folder.iterdir():
    if p.is_file():
        p.unlink()

input_file = input_files[0]

print(f'\n📁 Input: {input_file.name}')
print(f'⚡ Scale: {scale}×')
print('🧠 Model: RealESRGAN_x4plus')
print('⏳ Processing...')

command = [
    '/content/venv311/bin/python',
    '/content/realesrgan/inference_realesrgan.py',
    '-n', 'RealESRGAN_x4plus',
    '-i', str(input_file),
    '-o', str(output_folder),
            '--outscale', str(scale),
    '--model_path', '/content/realesrgan/weights/RealESRGAN_x4plus.pth'
]

result = subprocess.run(command, capture_output=True, text=True)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise Exception('❌ Upscaling failed.')

output_files = [p for p in output_folder.iterdir() if p.is_file()]

if not output_files:
    raise Exception('❌ No output generated.')

output_file = output_files[0]
img = Image.open(output_file)

print('🔥 OTAKU UPSCALE COMPLETE!')
print(f'📤 Output: {output_file.name}')
print(f'📐 Resolution: {img.size}')
print(f'🖼️ Format: {img.format}')

display(img)

In [ ]:
# ============================================
# 📥 DOWNLOAD RESULT
# ============================================

from google.colab import files
        
output_files = [p for p in output_folder.iterdir() if p.is_file()]

if output_files:
    files.download(str(output_files[0]))
else:
    print('❌ No output file found.')

## 🔥 OTAKU UPSCALER V1 ROADMAP

### Current
- ✅ Colab GPU
- ✅ Python 3.11 environment
- ✅ Real-ESRGAN AI model
- ✅ Image upload
- ✅ 2× / 4× upscale
- ✅ Preview
- ✅ Download

### Next
- ⬜ Topaz-inspired enhancement engine
- ⬜ Anime / Movie processing modes
- ⬜ Detail recovery
- ⬜ Denoise
- ⬜ Dehalo
- ⬜ Sharpen
- ⬜ Anti-Alias / DeBlur
- ⬜ Compression recovery

### V2
- ⬜ Video upscaling
- ⬜ CC pipeline
- ⬜ 60 FPS interpolation
- ⬜ Before CC → Upscale → CC → After CC workflow